# TrainLM TPU v5e-8 validation
Run this notebook from the TrainLM repository root on a TPU VM or supported TPU notebook. It loads an unchanged Hugging Face causal language model, validates packed `.bin` shards, builds TrainLM's task/optimizer/scheduler, and runs a smoke test followed by a measured run.
This is a staged validation run, not the 20B-token production launch. Record the evidence first; scale the step count only after the smoke and baseline gates are healthy.


## Prerequisites and fixed geometry
- TPU v5e-8 with the pinned profile in `constraints/tpu-xla-2.9.txt`.
- One or more TrainLM packed-shard manifests (`*.json`) next to their `.bin` payloads. The manifest supplies header, dtype, checksum, and vocabulary bounds; the reader will not guess them.
- Reference geometry: sequence length 2048, micro-batch 2 per device, 32 accumulation steps, 8 data-parallel replicas = 1,048,576 scheduled tokens per optimizer update.
- A direct Jupyter cell may see `world_size=1` because no worker fan-out has been launched. Set `TRAINLM_ALLOW_SINGLE_PROCESS=1` only for a functional smoke; it is not a v5e-8 throughput or MFU measurement.
- Set `TRAINLM_HF_REVISION` to an immutable Hub commit when loading pretrained weights. Keep `TRAINLM_TRUST_REMOTE_CODE=0` unless the repository has been reviewed.


In [ ]:
# Kaggle/Colab bootstrap: clone the repository in a separate cell if needed,
# then run this notebook from that checkout. Do not run git commands during
# an active training job.
import os
# Kaggle sometimes leaves an invalid single-host override behind. Remove it
# before importing torch_xla so PJRT can discover the TPU topology.
for _name in ("TPU_PROCESS_ADDRESSES", "JAX_TPU_PROCESS_ADDRESSES"):
    if os.environ.get(_name, "").strip().lower() == "local":
        os.environ.pop(_name, None)
from pathlib import Path

REPO_DIR = Path(os.environ.get("TRAINLM_REPO_DIR", "/kaggle/working/TrainLM")).resolve()
if (REPO_DIR / "pyproject.toml").is_file():
    os.chdir(REPO_DIR)
else:
    raise FileNotFoundError(f"TrainLM checkout not found: {REPO_DIR}")
print("working directory:", Path.cwd())


In [ ]:
# Install once per TPU environment, then restart the kernel.
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt


In [ ]:
import os
import time
import platform
from pathlib import Path
from types import SimpleNamespace

import torch
import transformers
import trainlm
import torch_xla
import torch_xla.runtime as xla_runtime

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_xla": getattr(torch_xla, "__version__", "unknown"),
    "transformers": transformers.__version__,
    "xla_device": str(torch_xla.device()),
    "world_size": int(xla_runtime.world_size()),
    "ordinal": int(xla_runtime.global_ordinal()),
})
WORLD_SIZE = int(xla_runtime.world_size())
ORDINAL = int(xla_runtime.global_ordinal())
ALLOW_SINGLE_PROCESS = os.environ.get("TRAINLM_ALLOW_SINGLE_PROCESS", "0") == "1"
if WORLD_SIZE not in (1, 8):
    raise RuntimeError(f"Unexpected Torch/XLA process count: {WORLD_SIZE}")
if WORLD_SIZE == 1 and not ALLOW_SINGLE_PROCESS:
    print("Coordinator process detected (world_size=1). Use the DP8 launch cell below for training; set TRAINLM_ALLOW_SINGLE_PROCESS=1 only for an in-process functional smoke.")


In [ ]:
SEQ_LEN = 2048
MICRO_BATCH_PER_DEVICE = 2
GRADIENT_ACCUMULATION_STEPS = 32
SMOKE_STEPS = 2
TRAIN_STEPS = int(os.environ.get("TRAINLM_TRAIN_STEPS", "100"))
CACHE_DIR = Path(os.environ.get("TRAINLM_XLA_CACHE", "jax_cache/trainlm_v5e8"))
OUTPUT_DIR = Path(os.environ.get("TRAINLM_OUTPUT", "runs/trainlm_v5e8"))
MANIFEST_DIR = Path(os.environ.get("TRAINLM_MANIFEST_DIR", "data/packed/train"))
DATA_MODE = os.environ.get("TRAINLM_DATA_MODE", "local")
HF_DATASET_REPO = os.environ.get("TRAINLM_DATASET_REPO", "LaughTaleAI/LaughLM-Tokenized-Fine")
HF_DATASET_REVISION = os.environ.get("TRAINLM_DATASET_REVISION", "")
HF_DATASET_ROOT = os.environ.get("TRAINLM_DATASET_ROOT", "laughlm-v1")
HF_SHARD_START = int(os.environ.get("TRAINLM_SHARD_START", "0"))
HF_SHARD_COUNT = int(os.environ.get("TRAINLM_SHARD_COUNT", "1"))
HF_MANIFEST_TEMPLATE = os.environ.get("TRAINLM_MANIFEST_TEMPLATE", "{root}/laughlm-v1_shard_{index:05d}.manifest.json")
MODEL_ID = os.environ.get("TRAINLM_MODEL_ID", "")
HF_REVISION = os.environ.get("TRAINLM_HF_REVISION", "")
TRUST_REMOTE_CODE = os.environ.get("TRAINLM_TRUST_REMOTE_CODE", "0") == "1"
EXPORT_HF = os.environ.get("TRAINLM_EXPORT_HF", "0") == "1"

assert SEQ_LEN >= 2 and MICRO_BATCH_PER_DEVICE >= 1
assert GRADIENT_ACCUMULATION_STEPS >= 1 and TRAIN_STEPS >= 1
print("scheduled tokens/update:", SEQ_LEN * MICRO_BATCH_PER_DEVICE * GRADIENT_ACCUMULATION_STEPS * WORLD_SIZE)


## DP8 launch (recommended)
The notebook kernel is only a coordinator and normally reports `world_size=1`. Run the worker script below to fan out eight PJRT processes; it constructs the model, validated reader, Trainer, optimizer, and XLA runtime inside each worker. Do not use the in-process cells below for throughput or MFU claims.


In [ ]:
import subprocess
import sys

LAUNCH_STEPS = int(os.environ.get("TRAINLM_LAUNCH_STEPS", str(SMOKE_STEPS)))
worker_cmd = [sys.executable, "scripts/trainlm_tpu_worker.py", "--max-steps", str(LAUNCH_STEPS), "--data-mode", DATA_MODE, "--manifest-dir", str(MANIFEST_DIR), "--dataset-repo", HF_DATASET_REPO, "--dataset-revision", HF_DATASET_REVISION, "--dataset-root", HF_DATASET_ROOT, "--shard-start", str(HF_SHARD_START), "--shard-count", str(HF_SHARD_COUNT), "--manifest-template", HF_MANIFEST_TEMPLATE]
print("launching DP8 worker:", " ".join(worker_cmd))
subprocess.run(worker_cmd, check=True)


## 1. Acquire the HF model
The provider uses `AutoConfig` and `AutoModelForCausalLM`; it does not copy or edit the model implementation. Leave `MODEL_ID` empty to construct the 135M Llama-shaped reference from config. Set it to any supported dense autoregressive Hub/local model for a compatibility run; the selected HF config remains authoritative.


In [ ]:
from trainlm.config import ModelSourceConfig
from trainlm.model import load_huggingface_causal_lm, explain_huggingface_compatibility

def make_source():
    if MODEL_ID:
        return ModelSourceConfig(
            provider="huggingface", initialization="pretrained",
            name_or_path=MODEL_ID, revision=HF_REVISION or None,
            trust_remote_code=TRUST_REMOTE_CODE, dtype="float32",
            use_safetensors=True,
        )
    return ModelSourceConfig(
        provider="huggingface", initialization="config", model_type="llama",
        dtype="float32",
        config_overrides={
            "vocab_size": 32064, "hidden_size": 1024,
            "intermediate_size": 2816, "num_hidden_layers": 8,
            "num_attention_heads": 8, "num_key_value_heads": 8,
            "max_position_embeddings": SEQ_LEN, "tie_word_embeddings": True,
        },
    )

source = make_source()
loaded = load_huggingface_causal_lm(source)
print(loaded.metadata.to_dict())
print("parameters:", sum(p.numel() for p in loaded.model.parameters()))
print(explain_huggingface_compatibility(loaded).explain())
print("Unknown architecture capabilities stay on the official HF implementation until evidence-backed transforms are selected.")


## 2. Validate packed `.bin` shards
`TRAINLM_DATA_MODE=hf` resolves the linked `LaughTaleAI/LaughLM-Tokenized-Fine` layout through the revision-pinned Hugging Face source. That path still requires a manifest for every `.bin`; a raw resolve URL or the pasted dataset summary is not enough to prove header, checksum, and token bounds. If the repository contains only raw `.bin` files, generate and publish TrainLM manifests first, or use `TRAINLM_DATA_MODE=local` with validated manifests.
The supplied metadata reports `microsoft/Phi-3.5-mini-instruct`, `uint16`, vocab 32011, and max token ID 32010. A model vocab of 32064 only bounds those IDs; it does not make a Llama tokenizer semantically equivalent. Use the matching tokenizer/model vocabulary mapping for a valid training comparison.
Validation scans token bounds and SHA-256 before opening memory maps. Keep the first TPU run single-process on the host (`num_workers=0`, `pin_memory=False`); increase host prefetch only after input-idle telemetry justifies it.


In [ ]:
from torch.utils.data import DataLoader, IterableDataset
from trainlm.data import (
    ContiguousPackedBatchReader, PackedBinaryShardManifest,
    PartitionedPackedBatchReader, plan_packed_batch_partition,
    validate_packed_binary_shard, HuggingFacePackedShardSource,
    HuggingFaceShardSourceConfig, HuggingFaceShardSpec,
)

def load_validated_shards(directory: Path):
    manifest_paths = sorted(directory.glob("*.json"))
    if not manifest_paths:
        raise FileNotFoundError(f"No shard manifests found in {directory}")
    shards = []
    for manifest_path in manifest_paths:
        manifest = PackedBinaryShardManifest.from_json(manifest_path.read_text(encoding="utf-8"))
        data_path = manifest_path.parent / Path(manifest.data_path)
        document_path = None
        if manifest.documents.path is not None:
            document_path = manifest_path.parent / Path(manifest.documents.path)
        validation = validate_packed_binary_shard(manifest, data_path, document_index_file=document_path)
        shards.append(SimpleNamespace(shard_id=manifest.shard_id, data_file=data_path, manifest=manifest, validation=validation))
        print(manifest.shard_id, manifest.token_count, "tokens: validated")
    return shards

class BatchIterable(IterableDataset):
    def __init__(self, reader): self.reader = reader
    def __iter__(self): yield from self.reader
    def __len__(self): return len(self.reader)

if DATA_MODE == "hf":
    if len(HF_DATASET_REVISION) != 40:
        raise ValueError("HF_DATASET_REVISION must be the 40-character commit SHA; mutable branches are rejected.")
    hf_source = HuggingFacePackedShardSource(HuggingFaceShardSourceConfig(
        repo_id=HF_DATASET_REPO, revision=HF_DATASET_REVISION,
        cache_dir=os.environ.get("HF_HOME", "/tmp/laughlm_hf_cache"),
        shards=tuple(HuggingFaceShardSpec(
            shard_id=f"laughlm-v1_shard_{index:05d}",
            manifest_path=HF_MANIFEST_TEMPLATE.format(root=HF_DATASET_ROOT, index=index),
        ) for index in range(HF_SHARD_START, HF_SHARD_START + HF_SHARD_COUNT)),
    ))
    shards = list(hf_source.resolve())
else:
    shards = load_validated_shards(MANIFEST_DIR)
reader = ContiguousPackedBatchReader(shards, batch_size=MICRO_BATCH_PER_DEVICE, sequence_length=SEQ_LEN)
partition = plan_packed_batch_partition(
    reader, split="train", seed=42, epoch=0,
    world_size=WORLD_SIZE, rank=ORDINAL,
    cross_shard_remainder="drop", host_remainder="drop",
)
partitioned_reader = PartitionedPackedBatchReader(reader, partition)
train_loader = DataLoader(BatchIterable(partitioned_reader), batch_size=None, num_workers=0, pin_memory=False)
print("rank batches:", len(partitioned_reader), "dropped tokens:", partition.dropped_token_count)


## 3. Build the XLA runtime and trainer
Parameters stay in FP32 and are autocast to BF16 for compute. `fused=False` is deliberate: CUDA fused AdamW is not the TPU/XLA optimizer path; XLA owns the device update. The persistent compilation cache is initialized by `XlaRuntime`.


In [ ]:
from trainlm.config import (
    CheckpointConfig, DatasetConfig, LossConfig, LoggingConfig, MonitoringConfig,
    OptimizationConfig, OptimizerConfig, ParallelismConfig, RuntimeConfig,
    SchedulerConfig, TrainConfig, TrainerConfig,
)
from trainlm.optimization import create_optimizer
from trainlm.runtime import XlaRuntime
from trainlm.tasks import CausalLMTask
from trainlm.training import Trainer, TrainerCallback, create_scheduler

class PrintMetrics(TrainerCallback):
    def __init__(self, runtime): self.runtime = runtime
    def on_metrics(self, state, control, metrics):
        if self.runtime.is_primary_process:
            print({k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()})

class ThroughputProbe(TrainerCallback):
    def __init__(self, runtime, warmup_steps=5):
        self.runtime, self.warmup_steps = runtime, warmup_steps
        self.start_time, self.start_tokens = None, None
        self.tokens_per_second = None
    def on_step_end(self, state, control):
        if state.step == self.warmup_steps:
            self.start_time, self.start_tokens = time.perf_counter(), state.tokens_seen
        elif self.start_time is not None and state.step > self.warmup_steps:
            elapsed = time.perf_counter() - self.start_time
            local_tokens = state.tokens_seen - self.start_tokens
            self.tokens_per_second = local_tokens * self.runtime.world_size / max(elapsed, 1e-9)
    def report(self):
        return {"steady_state_global_tokens_per_second": self.tokens_per_second}

def build_trainer(max_steps):
    loaded_run = load_huggingface_causal_lm(source)
    runtime = XlaRuntime(precision="bf16", cache_dir=CACHE_DIR, compile_training=False, collect_diagnostics=True)
    config = TrainConfig(
        model=source,
        dataset=DatasetConfig(sequence_length=SEQ_LEN, num_workers=0, pin_memory=False, prefetch_factor=2, persistent_workers=False, packing=True),
        loss=LossConfig(implementation="causal_lm", normalization="supervised_tokens", z_loss=1e-4, logits_chunk_size=4096),
        runtime=RuntimeConfig(device="xla", precision="bf16", strategy="replicated"),
        parallelism=ParallelismConfig(data=WORLD_SIZE),
        optimizations=OptimizationConfig(policy="auto", compile=False, allow_fallbacks=True, compilation_cache_dir=str(CACHE_DIR), accumulation_strategy="microstep"),
        optimizer=OptimizerConfig(learning_rate=2e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1, fused=False, mu_dtype="bfloat16", nu_dtype="float32"),
        scheduler=SchedulerConfig(name="wsd", horizon_tokens=20_000_000_000, warmup_fraction=0.01, stable_fraction=0.95, min_lr_ratio=0.05),
        trainer=TrainerConfig(max_steps=max_steps, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, max_grad_norm=1.0, seed=42),
        checkpoint=CheckpointConfig(output_dir=OUTPUT_DIR / "checkpoints"),
        logging=LoggingConfig(log_every_steps=10, output_dir=OUTPUT_DIR),
        monitoring=MonitoringConfig(enabled=True, compile_metrics=False, memory_metrics=False, training_integrity=False),
    )
    config.validate()
    optimizer = create_optimizer(loaded_run.model.parameters(), config.optimizer)
    scheduler = create_scheduler(optimizer, config.scheduler)
    task = CausalLMTask(ignore_index=config.loss.ignore_index, normalization=config.loss.normalization, z_loss=config.loss.z_loss, loss_implementation=config.loss.implementation)
    throughput = ThroughputProbe(runtime)
    trainer = Trainer(config=config, model=loaded_run.model, runtime=runtime, optimizer=optimizer, scheduler=scheduler, task=task, train_dataloader=train_loader, callbacks=[PrintMetrics(runtime), throughput])
    return loaded_run, runtime, trainer, throughput

print("Trainer factory ready; no TPU work has started yet.")


In [ ]:
# Optional SPMD experiment. Keep disabled for the first generic smoke.
# The replicated path is the controlled baseline. Enable only after it passes.
ENABLE_SPMD = False
if ENABLE_SPMD:
    from trainlm.runtime import LogicalMesh
    mesh = runtime.create_mesh(LogicalMesh({"data": runtime.world_size}))
    model = runtime.shard_model(model, mesh)
    print(mesh.logical.axis_sizes)


## 4. Smoke test
The smoke performs a few complete optimizer updates and catches model dispatch, fixed-shape input, causal loss, gradient accumulation, and XLA lifecycle errors. A fresh trainer/model is built for the measured run so its compile and timing evidence is not contaminated by the smoke.


In [ ]:
smoke_loaded, smoke_runtime, smoke_trainer, smoke_throughput = build_trainer(SMOKE_STEPS)
print("smoke start")
smoke_state = smoke_trainer.train()
print("smoke end:", smoke_state.phase.value, "steps:", smoke_state.step, "tokens:", smoke_state.tokens_seen)
print("smoke diagnostics:", smoke_runtime.diagnostics())
print("smoke throughput (available after warm-up):", smoke_throughput.report())


## 5. Measured training run
After the smoke succeeds, run this cell. For a clean benchmark, set `TRAINLM_TRAIN_STEPS` before starting the kernel (for example, 100–1000 for a short baseline). The first compilation/warm-up steps are not a steady-state throughput claim.


In [ ]:
loaded_run, runtime, trainer, throughput = build_trainer(TRAIN_STEPS)
print("training start")
state = trainer.train()
print("training end")
print({
    "phase": state.phase.value,
    "optimizer_steps": state.step,
    "tokens_seen": state.tokens_seen,
    "last_loss": state.loss,
    "learning_rate": state.learning_rate,
})
print("backend diagnostics:", runtime.diagnostics())
print("XLA diagnostics:", runtime.collect_diagnostics())
print("throughput evidence:", throughput.report())


In [ ]:
# Optional plain Transformers export. Only ordinal 0 writes.
if EXPORT_HF and runtime.is_primary_process:
    export_dir = OUTPUT_DIR / "hf_export"
    export_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(export_dir, safe_serialization=True)
    print("saved HF export to", export_dir)
else:
    print("HF export skipped; set TRAINLM_EXPORT_HF=1 to enable it on ordinal 0.")

# Exact-resume checkpoints require an application-owned writer/loader callback.
# trainer.save_checkpoint(...) intentionally raises until that callback is supplied;
# see docs/checkpoint/TPU_ROUND_TRIP.md.
reader.close()


## Acceptance checklist
1. All eight ordinals validate the same shard geometry and reach `finalized` in the smoke.
2. Record steady-state global tokens/s only after warm-up; also retain loss, compile count, CPU fallback counters, and runtime diagnostics.
3. Compare the run with `docs/benchmarks/M5_BASELINE.md`; do not claim MFU from a short smoke.
4. If the generic HF path is materially below the LaughLM baseline, preserve the evidence and stop before enabling SPMD or architecture capability transforms.
